# Sistema RAG agéntico con herramienta de recuperación jurídica

Integrantes:

- Pablo Cabrejos
- Martin Valencia
- Samuel Lopez
- Miguel Ortiz

## Objetivo y alcance

Esta etapa incorpora una **capa de decisión agéntica** a un sistema de recuperación aumentada por generación (RAG) orientado a consultas sobre derecho laboral individual colombiano. El propósito es determinar cuándo una pregunta requiere consultar el corpus jurídico antes de generar una respuesta, en lugar de ejecutar la recuperación de forma incondicional.

La solución utiliza un agente de **un paso de decisión**: un modelo de lenguaje propone si debe invocarse la herramienta `buscar_articulos_laborales` o si corresponde responder sin consultar el corpus. Cuando se activa la herramienta, se utiliza el recuperador avanzado C2 ya integrado en el proyecto. Para evitar pérdida de información jurídica, la recuperación recibe **la pregunta original completa**, sin reducirla a palabras clave ni reemplazarla por una reformulación del modelo.

La evaluación posterior compara el sistema agéntico con C2 bajo las mismas condiciones experimentales. Así es posible observar no solo la calidad de las respuestas, sino también las decisiones de uso de herramienta, los artículos recuperados y los costos de latencia.

## 1. Preparación del entorno de ejecución

La siguiente celda configura el entorno de Google Colab, descarga el repositorio de trabajo cuando todavía no está disponible y verifica las dependencias necesarias. También establece las rutas desde las que se cargarán los componentes del sistema y sus recursos. Esta preparación permite que el experimento parta de una estructura reproducible y evita duplicar manualmente el código de recuperación.


In [1]:
# Bootstrap exclusivo para Google Colab con GPU T4.
import os
import subprocess
import sys
from importlib.metadata import PackageNotFoundError, version as version_instalada
from pathlib import Path

from packaging.version import Version

REPO_URL = 'https://github.com/Bosnape/cabrejos-ortiz-valencia-lopez.git'
RAIZ = Path('/content/cabrejos-ortiz-valencia-lopez')
if not RAIZ.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(RAIZ)], check=True)
os.chdir(RAIZ)

# Compatibilidad con las funciones comunes copiadas desde rag_avanzado.ipynb.
EN_COLAB = True

# PEFT 0.20 rechaza torchao 0.10.0 del runtime actual de Colab.
try:
    torchao = version_instalada('torchao')
except PackageNotFoundError:
    torchao = None
if torchao is not None and Version(torchao) < Version('0.16.0'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

# Versiones del notebook avanzado de Pablo, más las de salida estructurada del agente.
pillow_fija = version_instalada('pillow')
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers==6.0.1', 'transformers==5.16.1', 'anthropic==1.4.0',
    'pandas==3.0.5', 'python-dotenv==1.2.3', 'peft==0.20.0', 'accelerate==1.14.0',
    'chromadb', 'rank_bm25', 'pydantic', f'pillow=={pillow_fija}',
], check=True)

requeridos = [
    Path('notebooks/evaluation/rubrica.json'), Path('notebooks/evaluation/eval_set.json'),
    Path('data/diccionario_articulos.csv'), Path('s04_lora_adapter/adapter_config.json'),
    Path('s04_lora_adapter/adapter_model.safetensors'),
]
faltantes = [str(ruta) for ruta in requeridos if not (RAIZ / ruta).is_file()]
if faltantes:
    raise FileNotFoundError('Faltan archivos del repositorio: ' + ', '.join(faltantes))
print('Repositorio:', RAIZ)
print('GPU:', __import__('torch').cuda.get_device_name(0))

Repositorio: /content/cabrejos-ortiz-valencia-lopez
GPU: Tesla T4


## 2. Integración del sistema de recuperación y evaluación

El agente se construye sobre la infraestructura RAG existente para traer la capa del agente al sistema que ya se tiene para la ayuda de casos jurídicos

La siguiente celda carga y ejecuta **únicamente los bloques necesarios** del notebook de recuperación avanzada. Esta integración conserva la configuración del recuperador C2, el modelo generador y el harness, sin ejecutar experimentos adicionales ni sobrescribir los resultados de otras pruebas. De este modo, la principal diferencia entre los sistemas comparados queda localizada en la capa que decide si debe utilizarse la herramienta.


In [2]:
import json

base_path = RAIZ / 'notebooks/rag/rag_avanzado.ipynb'
base = json.loads(base_path.read_text(encoding='utf-8'))

def ejecutar_celda_base(indice):
    codigo = ''.join(base['cells'][indice]['source'])
    exec(compile(codigo, f'{base_path.name}:celda_{indice}', 'exec'), globals())

# Constantes, funciones de corpus y corpus validado.
ejecutar_celda_base(2)
ejecutar_celda_base(4)
eval_set = cargar_eval_set(RAIZ)
corpus = cargar_corpus(RAIZ)
# El cross-encoder de C2 requiere el mismo texto con prefijo de fuente que usa rag_avanzado.ipynb.
textos_ce = (corpus['fuente'] + '. ' + corpus['texto']).tolist()
assert validar_eval_set(eval_set)

# Denso + BM25 + RRF + cross-encoder, plantilla y decoder.
for indice in (7, 8, 9, 11, 13, 14, 16):
    ejecutar_celda_base(indice)

# Regla de evaluación, juez y harness sin modificaciones.
for indice in (18, 19, 20, 21, 22, 23):
    ejecutar_celda_base(indice)

assert validar_normas(eval_set, corpus)
print(f'C2 y harness listos: {len(corpus)} artículos, {len(eval_set)} casos, dispositivo {device}.')

device: cuda Tesla T4


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Decoder listo: Qwen/Qwen2.5-1.5B-Instruct
Sistemas listos: ['A_ingenuo', 'B_hibrida', 'C1_denso_rerank', 'C2_hibrida_rerank', 'D_multiquery'] | avanzado: C2_hibrida_rerank
Normas esperadas válidas en los 13 casos.
Rúbrica v2.0.0
C2 y harness listos: 143 artículos, 13 casos, dispositivo cuda.


## 3. Diseño e implementación del agente con herramienta

La arquitectura separa tres responsabilidades: **decidir**, **recuperar** y **responder**. Primero, el modelo de lenguaje analiza la pregunta y produce una propuesta estructurada en JSON. Después, esa propuesta se valida mediante un esquema de Pydantic. Finalmente, según la acción validada, el sistema invoca el recuperador C2 o sigue la ruta sin herramienta.

La herramienta `buscar_articulos_laborales` funciona como interfaz entre el agente y el corpus jurídico. Aunque la propuesta del modelo pueda incluir una consulta sugerida, esta se registra solo con fines de trazabilidad: **la búsqueda efectiva utiliza siempre el texto original de la persona**. Esta decisión evita que una reformulación breve suprima circunstancias relevantes, como el tipo de vínculo laboral, una condición de protección especial o la intervención de una autoridad.

 La elección `no_tool` depende de la salida del modelo y de su validación estructural. Cuando la salida no puede interpretarse correctamente, se aplica una ruta de recuperación conservadora y se deja constancia de lo ocurrido. El límite de un paso impide ciclos indefinidos de invocación.

Además de la respuesta final, la implementación registra la acción propuesta, la acción ejecutada, la consulta enviada al recuperador y las latencias correspondientes. Estos datos permiten auditar el comportamiento del agente y distinguir errores de decisión de errores de recuperación o generación.


In [8]:
import time
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError, ConfigDict

NOMBRE_TOOL = 'buscar_articulos_laborales'
MAX_PASOS_AGENTE = 1

TOOLS = [{
    'name': NOMBRE_TOOL,
    'description': ('Busca hasta tres artículos en el corpus local de derecho laboral individual colombiano. '
                    'Úsala para toda consulta laboral, incluso si sospechas que el corpus no tiene la respuesta.'),
    'parameters': {
        'type': 'object',
        'properties': {'consulta': {'type': 'string', 'description': 'Consulta original completa del usuario, sin reformulaciones.'}},
        'required': ['consulta'],
        'additionalProperties': False,
    },
}]

class ArgumentosHerramienta(BaseModel):
    consulta: Optional[str] = None
    model_config = ConfigDict(extra='forbid')

class DecisionHerramienta(BaseModel):
    action: Literal['buscar_articulos_laborales', 'no_tool']
    arguments: ArgumentosHerramienta

    model_config = ConfigDict(extra='forbid')

# Pydantic es la definición única del contrato; de ella sale el JSON Schema.
ESQUEMA_DECISION = (DecisionHerramienta.model_json_schema()
                    if hasattr(DecisionHerramienta, 'model_json_schema')
                    else DecisionHerramienta.schema())

SISTEMA_AGENTE = """
Eres el controlador de herramientas de un asistente especializado
en derecho laboral individual colombiano.

Tu única tarea es decidir si una pregunta requiere utilizar
la herramienta buscar_articulos_laborales.

No debes responder la pregunta, redactar asesoría jurídica
ni generar explicaciones adicionales.

REGLAS DE DECISIÓN:

1. Utiliza buscar_articulos_laborales cuando la consulta esté
   relacionada con trabajo, empleados, contratos laborales,
   salarios, despidos, incapacidades, discriminación laboral,
   protección laboral o derechos de los trabajadores.

2. Si existen dudas sobre si la consulta pertenece al ámbito
   laboral, utiliza buscar_articulos_laborales.

3. Utiliza no_tool únicamente cuando el asunto sea claramente
   ajeno al derecho laboral, como una herencia familiar
   sin relación con un vínculo de trabajo.

FORMATO OBLIGATORIO DE RESPUESTA:

Debes devolver exclusivamente un objeto JSON válido.

El objeto debe contener exactamente dos propiedades:
- "action"
- "arguments"

La propiedad "action" solo puede contener:
- "buscar_articulos_laborales"
- "no_tool"

La propiedad "arguments" debe ser siempre un objeto JSON.

SI DEBES UTILIZAR LA HERRAMIENTA:

Devuelve exactamente esta estructura:

{
  "action": "buscar_articulos_laborales",
  "arguments": {
    "consulta": "PREGUNTA ORIGINAL COMPLETA DEL USUARIO"
  }
}

Copia la pregunta original completa en arguments.consulta.
No la resumas, reformules, traduzcas ni elimines información.
Conserva los hechos, fechas, circunstancias y términos jurídicos.

SI NO DEBES UTILIZAR LA HERRAMIENTA:

Devuelve exactamente esta estructura:

{
  "action": "no_tool",
  "arguments": {}
}

RESTRICCIONES:

- No escribas texto antes ni después del JSON.
- No utilices bloques Markdown.
- No agregues propiedades adicionales.
- No coloques "consulta" fuera de "arguments".
- No inventes acciones diferentes de las dos permitidas.
- No respondas la pregunta jurídica.

EJEMPLOS DE DECISIÓN:

- Despido durante el embarazo:
  buscar_articulos_laborales.

- Renuncia forzada de una persona con discapacidad:
  buscar_articulos_laborales.

- Repartición de una herencia familiar sin relación laboral:
  no_tool.

Devuelve únicamente el objeto JSON correspondiente
a la pregunta recibida.
""".strip()

def buscar_articulos_laborales(consulta):
    if not isinstance(consulta, str) or not consulta.strip():
        raise ValueError('La consulta de la herramienta debe ser texto no vacío.')
    indices = recuperar_C2(consulta.strip())
    return referencias_de(indices, corpus)

def extraer_primer_json(texto):
    """Obtiene el primer objeto JSON aun si Qwen agrega texto o fences."""
    if not isinstance(texto, str):
        return None
    limpio = texto.replace('```json', '').replace('```JSON', '').replace('```', '').strip()
    decoder = json.JSONDecoder()
    for posicion, caracter in enumerate(limpio):
        if caracter != '{':
            continue
        try:
            valor, _ = decoder.raw_decode(limpio[posicion:])
            if isinstance(valor, dict):
                return valor
        except json.JSONDecodeError:
            continue
    return None

def parsear_decision(texto, pregunta):
    """Separa la propuesta del modelo de la acción efectiva; nunca usa palabras clave."""
    propuesta = extraer_primer_json(texto)
    if propuesta is None:
        return {'action': 'fallback', 'consulta': pregunta, 'motivo': 'json_invalido',
                'accion_propuesta': None, 'consulta_propuesta': None}
    try:
        validada = (DecisionHerramienta.model_validate(propuesta)
                    if hasattr(DecisionHerramienta, 'model_validate')
                    else DecisionHerramienta.parse_obj(propuesta))
    except ValidationError:
        return {'action': 'fallback', 'consulta': pregunta, 'motivo': 'schema_invalido',
                'accion_propuesta': propuesta.get('action'),
                'consulta_propuesta': (propuesta.get('arguments') or {}).get('consulta')
                if isinstance(propuesta.get('arguments'), dict) else None}
    accion = validada.action
    consulta_propuesta = validada.arguments.consulta
    if accion == 'no_tool':
        return {'action': 'no_tool', 'consulta': None, 'motivo': 'modelo',
                'accion_propuesta': accion, 'consulta_propuesta': consulta_propuesta}
    # No usamos la reformulación del LLM: incluso una aparentemente buena puede perder hechos.
    return {'action': NOMBRE_TOOL, 'consulta': pregunta, 'motivo': 'modelo_consulta_original',
            'accion_propuesta': accion, 'consulta_propuesta': consulta_propuesta}

def propuesta_cumple_schema(texto):
    propuesta = extraer_primer_json(texto)
    if propuesta is None:
        return False
    try:
        if hasattr(DecisionHerramienta, 'model_validate'):
            DecisionHerramienta.model_validate(propuesta)
        else:
            DecisionHerramienta.parse_obj(propuesta)
        return True
    except ValidationError:
        return False

def decidir_herramienta(pregunta):
    propuesta = generar_texto(SISTEMA_AGENTE, f'Pregunta del usuario:\n{pregunta}', 96)
    intentos_formato = 1
    if not propuesta_cumple_schema(propuesta):
        sistema_reintento = SISTEMA_AGENTE + """

Tu respuesta anterior no cumplió el contrato.

Debes devolver exactamente una de estas estructuras:

Para buscar:
{"action":"buscar_articulos_laborales",
 "arguments":{"consulta":"PREGUNTA ORIGINAL COMPLETA"}}

Para no buscar:
{"action":"no_tool","arguments":{}}

No pongas consulta fuera de arguments.
No agregues texto explicativo.
"""
        propuesta = generar_texto(sistema_reintento, f'Pregunta del usuario:\n{pregunta}', 96)
        intentos_formato = 2
    decision = parsear_decision(propuesta, pregunta)
    decision['propuesta_modelo'] = propuesta
    decision['intentos_formato'] = intentos_formato
    return decision

def responder_sin_contexto():
    resumen = ('La consulta está fuera del alcance del corpus local de derecho laboral colombiano; '
               'por eso no se recuperaron artículos para responderla.')
    return armar_respuesta([], resumen)

def sistema_final(pregunta):
    """Interfaz común para el harness: texto y atributos de contexto/latencia."""
    if MAX_PASOS_AGENTE != 1:
        raise RuntimeError('Este experimento fija un solo paso de agente.')
    inicio_agente = time.perf_counter()
    decision = decidir_herramienta(pregunta)
    sistema_final.latencia_agente = time.perf_counter() - inicio_agente
    inicio_recuperacion = time.perf_counter()
    if decision['action'] in (NOMBRE_TOOL, 'fallback'):
        contexto = buscar_articulos_laborales(decision['consulta'])
        decision['tool_ejecutada'] = NOMBRE_TOOL
    else:
        contexto = []
        decision['tool_ejecutada'] = None
    sistema_final.latencia_recuperacion = time.perf_counter() - inicio_recuperacion
    sistema_final.ultimo_contexto = contexto
    sistema_final.consultas_reformuladas = []
    sistema_final.traza_agente = decision
    sistema_final.historial.append({
        'pregunta': pregunta, 'accion_final': decision['action'], 'motivo_agente': decision['motivo'],
        'herramienta_ejecutada': decision['tool_ejecutada'],
        'latencia_agente_seg': sistema_final.latencia_agente,
        'latencia_recuperacion_seg': sistema_final.latencia_recuperacion,
        'propuesta_modelo': decision['propuesta_modelo'],
        'accion_propuesta': decision['accion_propuesta'],
        'consulta_propuesta': decision['consulta_propuesta'],
        'consulta_ejecutada': decision['consulta'],
    })
    return responder(pregunta, contexto, generar_texto) if contexto else responder_sin_contexto()

sistema_final.ultimo_contexto = []
sistema_final.latencia_recuperacion = 0.0
sistema_final.latencia_agente = 0.0
sistema_final.consultas_reformuladas = []
sistema_final.traza_agente = {}
sistema_final.historial = []
print('Herramienta y sistema_final listos:', NOMBRE_TOOL)

Herramienta y sistema_final listos: buscar_articulos_laborales


## 4. Pruebas funcionales de selección de herramienta

Antes de ejecutar la evaluación completa, se realizan tres pruebas de comportamiento que representan decisiones esperadas distintas. Las dos primeras plantean situaciones de derecho laboral —un despido durante el embarazo y una renuncia exigida a una persona con discapacidad— en las que resulta pertinente consultar normas del corpus. La tercera plantea una controversia sobre herencia familiar, fuera del ámbito laboral del sistema, para comprobar la ruta sin recuperación.

Estas pruebas permiten observar si el agente distingue las consultas que requieren apoyo documental de las que no corresponden al dominio de la herramienta. Para cada pregunta se presentan la decisión adoptada, las referencias obtenidas cuando procede y el tiempo empleado. Las decisiones son **resultados del modelo**, no reglas impuestas a partir de las palabras de los ejemplos; por tanto, deben revisarse después de ejecutar la celda.


In [9]:
pruebas_agente = [
    'Me despidieron estando embarazada y no hubo autorización del Ministerio. ¿Qué normas debo revisar?',
    'Tengo una discapacidad y mi empleador quiere obligarme a firmar una renuncia. ¿Qué protección laboral existe?',
    'Mis hermanos discuten cómo repartir la herencia de mi padre. ¿Qué debe hacer la familia?',
]

for pregunta in pruebas_agente:
    respuesta = sistema_final(pregunta)
    print('\nPregunta:', pregunta)
    print('Traza:', json.dumps(sistema_final.traza_agente, ensure_ascii=False))
    print('Artículos:', [r['cita'] for r in sistema_final.ultimo_contexto])
    print('Latencia agente:', round(sistema_final.latencia_agente, 2), 's')
    print('Respuesta:', respuesta[:500])


Pregunta: Me despidieron estando embarazada y no hubo autorización del Ministerio. ¿Qué normas debo revisar?
Traza: {"action": "buscar_articulos_laborales", "consulta": "Me despidieron estando embarazada y no hubo autorización del Ministerio. ¿Qué normas debo revisar?", "motivo": "modelo_consulta_original", "accion_propuesta": "buscar_articulos_laborales", "consulta_propuesta": "Despidos por embarazo, autorización del Ministerio, derecho a la seguridad social", "propuesta_modelo": "```json\n{\n  \"action\": \"buscar_articulos_laborales\",\n  \"arguments\": {\n    \"consulta\": \"Despidos por embarazo, autorización del Ministerio, derecho a la seguridad social\"\n  }\n}\n```", "intentos_formato": 1, "tool_ejecutada": "buscar_articulos_laborales"}
Artículos: ['Constitución Política Art. 43', 'Ley 50 de 1990 Art. 35', 'Ley 361 de 1997 Art. 26']
Latencia agente: 2.76 s
Respuesta: Referencias recuperadas:
1. Constitución Política Art. 43: La mujer y el hombre tienen iguales derechos y opor

## 5. Evaluación comparativa y generación de evidencias

La validación experimental utiliza el mismo conjunto de **13 casos** y el mismo mecanismo de evaluación para dos configuraciones: el recuperador avanzado C2 como sistema de control y el sistema final con decisión agéntica. Ambas configuraciones se ejecutan en la misma sesión para reducir diferencias derivadas del entorno, los modelos cargados o la configuración del juez.

La comparación considera la recuperación de normas esperadas mediante **recall@3**, la valoración del juez, los aciertos de dominio y la latencia. En el caso del agente, el registro detallado incorpora también la propuesta de acción, el uso efectivo de la herramienta y la consulta realmente ejecutada. Esto permite identificar, por ejemplo, una abstención indebida ante una consulta laboral o una recuperación que no aporta normas pertinentes.

La siguiente celda genera los archivos CSV de resultados agregados, resultados por caso y comparación entre configuraciones. Estos archivos constituyen la evidencia cuantitativa del experimento y deben analizarse **después de ejecutar la versión actual del notebook**. No se consideran válidas para esta implementación las métricas obtenidas con versiones anteriores del agente.


In [10]:
METADATOS_FINAL = {
    'modelo_embeddings': EMBED_MODEL,
    'modelo_cross_encoder': MODELO_CE,
    'modelo_generador': GEN_MODEL,
    'modelo_juez': JUEZ_MODEL,
    'herramienta': NOMBRE_TOOL,
    'max_pasos_agente': MAX_PASOS_AGENTE,
    'top_k': TOP_K,
    'seed': SEED,
    'version_rubrica': RUBRICA_VERSION,
}

# Control obligatorio: C2 y el agente se miden aquí, en la misma sesión, GPU, modelos y juez.
METADATOS_C2_CONTROL = {**METADATOS_FINAL, 'sistema': 'C2_hibrida_rerank', 'herramienta': 'ninguna'}
resultado_c2_control = harness(eval_set, sistema_rag_avanzado, juez, metadatos=METADATOS_C2_CONTROL)

# El historial solo debe contener las 13 decisiones de esta corrida final.
sistema_final.historial = []
resultado_final = harness(eval_set, sistema_final, juez, metadatos=METADATOS_FINAL)

# El harness común no se modifica. Se añaden trazas solo al detalle de esta corrida.
trazas_harness = sistema_final.historial[-len(eval_set):]
assert len(trazas_harness) == len(resultado_final['detalle'])
for fila_detalle, traza in zip(resultado_final['detalle'], trazas_harness):
    fila_detalle.update({
        'accion_agente': traza['accion_final'],
        'motivo_agente': traza['motivo_agente'],
        'herramienta_ejecutada': traza['herramienta_ejecutada'],
        'latencia_agente_seg': traza['latencia_agente_seg'],
        'propuesta_agente': traza['propuesta_modelo'],
        'accion_propuesta': traza['accion_propuesta'],
        'consulta_propuesta': traza['consulta_propuesta'],
        'consulta_ejecutada': traza['consulta_ejecutada'],
    })

latencia_agente_mediana = float(pd.DataFrame(trazas_harness)['latencia_agente_seg'].median())
comparacion_misma_sesion = pd.DataFrame([
    {'sistema': 'C2_control', **resultado_c2_control['resumen']},
    {'sistema': 'agente_final', **resultado_final['resumen']},
])
display(comparacion_misma_sesion)
print(f'Latencia mediana de decisión del agente: {latencia_agente_mediana:.3f} s')

salida = RAIZ / 'notebooks/rag/resultados/final'
salida.mkdir(parents=True, exist_ok=True)
escribir_scorecards(salida, 'c2_control_misma_sesion', resultado_c2_control)
escribir_scorecards(salida, 'sistema_final', resultado_final)
comparacion_misma_sesion.to_csv(salida / 'comparacion_c2_vs_agente_misma_sesion.csv', index=False)
print('CSV guardados en:', salida)

from google.colab import files
for archivo in (
    'scorecard_c2_control_misma_sesion.csv',
    'scorecard_c2_control_misma_sesion_detalle.csv',
    'scorecard_sistema_final.csv',
    'scorecard_sistema_final_detalle.csv',
    'comparacion_c2_vs_agente_misma_sesion.csv',
):
    files.download(str(salida / archivo))

M2-01: recall@3=0.0 juez=4.0 acierto=False
M2-02: recall@3=0.0 juez=1.0 acierto=False
M2-03: recall@3=0.0 juez=2.0 acierto=False
M2-04: recall@3=0.0 juez=1.0 acierto=False
M2-05: recall@3=0.0 juez=1.0 acierto=False
M2-06: recall@3=0.5 juez=2.0 acierto=False
M2-07: recall@3=0.0 juez=2.5 acierto=False
M2-08: recall@3=0.0 juez=1.5 acierto=False
M2-09: recall@3=0.5 juez=3.0 acierto=False
M2-10: recall@3=0.0 juez=1.0 acierto=False
M2-11: recall@3=nan juez=3.5 acierto=False
M2-12: recall@3=nan juez=4.0 acierto=True
M2-13: recall@3=1.0 juez=4.0 acierto=True
M2-01: recall@3=0.0 juez=4.0 acierto=False
M2-02: recall@3=0.0 juez=2.0 acierto=False
M2-03: recall@3=0.0 juez=2.0 acierto=False
M2-04: recall@3=0.0 juez=1.0 acierto=False
M2-05: recall@3=0.0 juez=1.0 acierto=False
M2-06: recall@3=0.5 juez=2.0 acierto=False
M2-07: recall@3=0.0 juez=2.0 acierto=False
M2-08: recall@3=0.0 juez=1.0 acierto=False
M2-09: recall@3=0.5 juez=3.0 acierto=False
M2-10: recall@3=0.0 juez=1.0 acierto=False
M2-11: recall

,sistema,total,recall_3_medio,juez_promedio,aciertos,latencia_mediana_seg,latencia_recuperacion_mediana_seg,pares_validos,diferencias_entre_ordenes,delta_medio,delta_absoluto_medio,indeterminados
0,C2_control,13,0.181818,2.346154,2,6.153358,0.748013,13,4,-0.230769,0.538462,0
1,agente_final,13,0.181818,2.346154,2,8.831397,0.700102,13,3,0.076923,0.384615,0


Latencia mediana de decisión del agente: 3.105 s
CSV guardados en: /content/cabrejos-ortiz-valencia-lopez/notebooks/rag/resultados/final


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 6. Evaluación diagnóstica de selección de herramientas (20 consultas)

Esta evaluación complementaria mide únicamente la capacidad del controlador para decidir cuándo recuperar artículos laborales. No sustituye el conjunto oficial de 13 casos, no modifica el corpus ni el recuperador C2 y no interviene en las métricas RAGAS o del juez jurídico.

Se construye un conjunto de 20 consultas etiquetadas antes de ejecutar el modelo: diez relacionadas con derecho laboral individual colombiano y diez ajenas a ese dominio. Se incluyen casos directos y fronterizos para observar si el agente distingue el asunto central de la pregunta, en lugar de depender de palabras aisladas. Las etiquetas corresponden al alcance definido para la herramienta y no representan una clasificación jurídica exhaustiva.

En cada caso se invoca exclusivamente decidir_herramienta(pregunta). Se registra la acción esperada, la acción producida, si hubo un fallo de formato, el número de intentos, el tiempo de decisión y la coincidencia con la etiqueta. La prueba no ejecuta buscar_articulos_laborales(), de modo que la latencia medida corresponde al controlador y no al sistema completo.

Las métricas incluyen exactitud, precisión y sensibilidad de la decisión de búsqueda, especificidad para las consultas fuera de dominio, falsos positivos y falsos negativos, además de tiempos y reintentos. Un fallback se contabiliza como decisión no válida, sin asumir que equivale a buscar o abstenerse. Esto evita atribuir al modelo decisiones que en realidad proceden de reglas de contingencia.

El conjunto es pequeño y diseñado manualmente, por lo que sus resultados constituyen un diagnóstico funcional, no una estimación estadística generalizable. Se conservan los registros individuales en CSV para revisar los errores antes de redactar conclusiones.

In [11]:
# Prueba diagnóstica del CONTROLADOR: no ejecuta recuperar_C2 ni responder().
import csv
import statistics
from pathlib import Path

CASOS_DECISION = [
    # Consultas que SÍ requieren recuperación laboral.
    ('D01', 'laboral', 'Me despidieron estando embarazada y no hubo autorización del Ministerio. ¿Qué normas debo revisar?', NOMBRE_TOOL),
    ('D02', 'laboral', 'Mi empleador quiere obligarme a renunciar porque tengo una discapacidad. ¿Qué protección existe?', NOMBRE_TOOL),
    ('D03', 'laboral', 'No me pagaron las horas extras de los últimos tres meses. ¿Qué derechos tengo?', NOMBRE_TOOL),
    ('D04', 'laboral', 'Tengo contrato a término fijo y no me avisaron de su terminación. ¿Qué reglas aplican?', NOMBRE_TOOL),
    ('D05', 'laboral', 'La empresa no consignó mis cesantías. ¿Qué puedo reclamar?', NOMBRE_TOOL),
    ('D06', 'laboral', 'Tuve un accidente de tránsito mientras hacía entregas ordenadas por mi jefe. ¿Puede ser accidente laboral?', NOMBRE_TOOL),
    ('D07', 'laboral', 'Mi empleador me exige firmar un contrato de arrendamiento de vivienda como condición para conservar el empleo. ¿Es legal?', NOMBRE_TOOL),
    ('D08', 'laboral', 'Trabajo desde casa y no me reconocen los descansos ni la jornada. ¿Qué normas aplican?', NOMBRE_TOOL),
    ('D09', 'laboral', 'Me incapacitaron por una enfermedad y la empresa amenaza con despedirme. ¿Qué protección tengo?', NOMBRE_TOOL),
    ('D10', 'laboral', 'Mi padre falleció y necesito saber si mi empleador debe concederme licencia por luto.', NOMBRE_TOOL),
    # Consultas que NO requieren recuperación en el corpus laboral individual.
    ('D11', 'no_laboral', 'Mis hermanos discuten cómo repartir la herencia de mi padre. ¿Qué debe hacer la familia?', 'no_tool'),
    ('D12', 'no_laboral', 'Choqué mi carro particular un domingo mientras paseaba. ¿Cómo reclamo al seguro?', 'no_tool'),
    ('D13', 'no_laboral', 'Mi arrendador quiere subir el canon de mi apartamento. ¿Cuánto puede aumentar?', 'no_tool'),
    ('D14', 'no_laboral', 'Quiero tramitar el divorcio y definir la custodia de mis hijos. ¿Qué hago?', 'no_tool'),
    ('D15', 'no_laboral', 'Compré un celular defectuoso. ¿Cómo solicito la garantía al vendedor?', 'no_tool'),
    ('D16', 'no_laboral', 'Mi vecino construyó un muro dentro de mi terreno. ¿Qué puedo hacer?', 'no_tool'),
    ('D17', 'no_laboral', 'Necesito pagar el impuesto predial de una casa que heredé. ¿Cómo se liquida?', 'no_tool'),
    ('D18', 'no_laboral', 'Quiero constituir una sociedad para invertir en inmuebles. ¿Qué documentos necesito?', 'no_tool'),
    ('D19', 'no_laboral', 'Mi vehículo recibió una fotomulta mientras iba de vacaciones. ¿Cómo la impugno?', 'no_tool'),
    ('D20', 'no_laboral', 'Mi familia quiere vender una vivienda y repartir el dinero entre los herederos. ¿Qué trámite corresponde?', 'no_tool'),
]

assert len(CASOS_DECISION) == 20
assert len({caso[0] for caso in CASOS_DECISION}) == 20
assert sum(c[3] == NOMBRE_TOOL for c in CASOS_DECISION) == 10

resultados_decision = []
for caso_id, categoria, pregunta, esperado in CASOS_DECISION:
    inicio = time.perf_counter()
    try:
        decision = decidir_herramienta(pregunta)
        accion = decision.get('action', 'fallback')
        motivo = decision.get('motivo', '')
        intentos = decision.get('intentos_formato')
        propuesta = decision.get('propuesta_modelo', '')
        error = ''
    except Exception as exc:
        accion, motivo, intentos, propuesta = 'error', 'excepcion', None, ''
        error = f'{type(exc).__name__}: {str(exc)[:250]}'
    duracion = time.perf_counter() - inicio
    valido = accion in (NOMBRE_TOOL, 'no_tool')
    fila = {
        'id': caso_id, 'categoria': categoria, 'pregunta': pregunta,
        'accion_esperada': esperado, 'accion_agente': accion,
        'decision_valida': valido, 'acierto': valido and accion == esperado,
        'motivo': motivo, 'intentos_formato': intentos,
        'latencia_decision_seg': round(duracion, 4),
        'propuesta_modelo': propuesta, 'error': error,
    }
    resultados_decision.append(fila)
    print(f"{caso_id} [{categoria}] esperado={esperado} obtenido={accion} "
          f"acierto={fila['acierto']} intentos={intentos} tiempo={duracion:.2f}s")

# Positivo = se requiere la herramienta. Fallback/error = decisión inválida.
tp = sum(r['accion_esperada'] == NOMBRE_TOOL and r['accion_agente'] == NOMBRE_TOOL for r in resultados_decision)
fn = sum(r['accion_esperada'] == NOMBRE_TOOL and r['accion_agente'] == 'no_tool' for r in resultados_decision)
fp = sum(r['accion_esperada'] == 'no_tool' and r['accion_agente'] == NOMBRE_TOOL for r in resultados_decision)
tn = sum(r['accion_esperada'] == 'no_tool' and r['accion_agente'] == 'no_tool' for r in resultados_decision)
invalidos = sum(not r['decision_valida'] for r in resultados_decision)
correctos = tp + tn

def division_segura(numerador, denominador):
    return round(numerador / denominador, 4) if denominador else None

metricas_decision = {
    'total': len(resultados_decision),
    'correctos': correctos,
    'exactitud_total': division_segura(correctos, len(resultados_decision)),
    'precision_busqueda': division_segura(tp, tp + fp),
    'sensibilidad_laboral': division_segura(tp, 10),
    'especificidad_no_laboral': division_segura(tn, 10),
    'verdaderos_positivos': tp,
    'verdaderos_negativos': tn,
    'falsos_positivos': fp,
    'falsos_negativos': fn,
    'decisiones_invalidas': invalidos,
    'tasa_decisiones_invalidas': division_segura(invalidos, len(resultados_decision)),
    'llamadas_potenciales_evitas_correctamente': tn,
    'latencia_decision_mediana_seg': round(statistics.median(r['latencia_decision_seg'] for r in resultados_decision), 4),
    'latencia_decision_media_seg': round(statistics.mean(r['latencia_decision_seg'] for r in resultados_decision), 4),
    'casos_con_reintento': sum((r['intentos_formato'] or 0) > 1 for r in resultados_decision),
}

print('\n=== RESUMEN DIAGNÓSTICO DE SELECCIÓN ===')
for clave, valor in metricas_decision.items():
    print(f'{clave}: {valor}')
print('\nCASOS ERRÓNEOS O INVÁLIDOS:')
for r in resultados_decision:
    if not r['acierto']:
        print(f"  {r['id']}: esperado={r['accion_esperada']} obtenido={r['accion_agente']} motivo={r['motivo']}")

# Guardar evidencias separadas del scorecard oficial de 13 casos.
SALIDA_DIAGNOSTICO = Path('/content/diagnostico_agente') if Path('/content').exists() else Path.cwd() / 'diagnostico_agente'
SALIDA_DIAGNOSTICO.mkdir(parents=True, exist_ok=True)
ruta_detalle = SALIDA_DIAGNOSTICO / 'diagnostico_decisiones_20_casos.csv'
ruta_resumen = SALIDA_DIAGNOSTICO / 'diagnostico_decisiones_resumen.csv'
with ruta_detalle.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(resultados_decision[0]))
    writer.writeheader()
    writer.writerows(resultados_decision)
with ruta_resumen.open('w', encoding='utf-8-sig', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(metricas_decision))
    writer.writeheader()
    writer.writerow(metricas_decision)
print('\nArchivos:', ruta_detalle, ruta_resumen)


D01 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=3.42s
D02 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=2.28s
D03 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=2.23s
D04 [laboral] esperado=buscar_articulos_laborales obtenido=no_tool acierto=False intentos=1 tiempo=1.50s
D05 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=2.68s
D06 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=2.69s
D07 [laboral] esperado=buscar_articulos_laborales obtenido=buscar_articulos_laborales acierto=True intentos=1 tiempo=2.68s
D08 [laboral] esperado=buscar_articulos_laborales obtenido=no_tool acierto=False intentos=1 tiempo=1.54s
D09 [laboral] esperado=buscar_articulos_laborales ob

## Análisis y conclusiones del agente de herramientas

### Desempeño del agente como controlador de herramientas

Con el propósito de evaluar de manera independiente la capacidad del agente para determinar cuándo utilizar la herramienta de recuperación jurídica, se desarrolló una prueba diagnóstica de 20 consultas, distribuidas equitativamente entre preguntas relacionadas con el derecho laboral y preguntas ajenas a este dominio.

El agente clasificó correctamente 18 de las 20 consultas, alcanzando una exactitud del 90 %. En particular, identificó correctamente las diez consultas no laborales, obteniendo una especificidad del 100 %, mientras que reconoció ocho de las diez consultas laborales, con una sensibilidad del 80 %. Los dos errores correspondieron a falsos negativos, es decir, consultas laborales para las cuales el agente decidió no utilizar la herramienta de recuperación.

Adicionalmente, todas las decisiones se generaron en el primer intento, sin errores de formato JSON ni necesidad de reintentos. La latencia mediana de la etapa de decisión fue de 1,521 segundos, lo que evidencia un funcionamiento estable del mecanismo de generación y validación estructurada durante esta prueba.

Estos resultados muestran un comportamiento preliminar favorable del agente para discriminar entre consultas relevantes y no relevantes. No obstante, los falsos negativos identificados representan una limitación importante, dado que pueden impedir la recuperación de información normativa necesaria para responder consultas laborales válidas.

### Comparación entre el sistema RAG avanzado y el sistema con agente

La evaluación comparativa realizada sobre los 13 casos oficiales no evidenció mejoras en las métricas agregadas de recuperación ni en la calidad de las respuestas al incorporar el agente. Ambos sistemas obtuvieron un recall@3 promedio de 0,1818, una calificación media del juez de 2,346 sobre 5 y dos aciertos de dominio.

En términos de rendimiento, la latencia mediana total aumentó de 6,15 segundos en el sistema C2 a 8,83 segundos en el sistema con agente, lo que representa un incremento aproximado del 43,5 %. Este comportamiento se relaciona con el costo adicional de ejecutar una etapa de decisión antes de determinar si es necesario consultar el recuperador.

Los resultados indican que la incorporación del agente no mejora directamente la recuperación de artículos ni la generación de respuestas, ya que ambos sistemas comparten el mismo mecanismo de recuperación avanzada. Por consiguiente, las limitaciones observadas en la recuperación normativa continúan afectando al sistema final.

Sin embargo, el diagnóstico independiente mostró que el agente tiene potencial para evitar consultas innecesarias al recuperador cuando identifica preguntas ajenas al derecho laboral. Este beneficio todavía no puede considerarse una mejora demostrada en la eficiencia global, pues debe contrastarse con el tiempo adicional requerido para tomar cada decisión y con el costo de las respuestas que efectivamente se evitan.

### Limitaciones de la evaluación

La principal limitación del diagnóstico es el tamaño reducido de la muestra, compuesta por únicamente 20 consultas. Aunque la exactitud obtenida fue del 90 %, este resultado no permite garantizar que el agente mantenga el mismo comportamiento ante un conjunto más amplio y diverso de preguntas.

Asimismo, la distribución equilibrada de consultas laborales y no laborales no necesariamente representa las condiciones reales de uso del sistema. Se requiere ampliar la evaluación con preguntas ambiguas, consultas multidominio, variaciones de redacción y casos que incluyan términos jurídicos o cotidianos que puedan inducir decisiones incorrectas.

Otra limitación corresponde a la presencia de dos falsos negativos. Este tipo de error es especialmente relevante porque provoca que el sistema omita la búsqueda de artículos aun cuando la pregunta pertenece al dominio jurídico definido.

Por otra parte, el bajo recall@3 observado en la evaluación oficial sugiere limitaciones importantes en la recuperación normativa. Sin embargo, los resultados disponibles no permiten determinar si estas dificultades se originan exclusivamente en el recuperador avanzado, en la cobertura del corpus o en otros componentes del sistema RAG.

Finalmente, la prueba diagnóstica evaluó la calidad y la latencia de la decisión del agente, pero no midió directamente el ahorro neto de recursos computacionales ni de tiempo de respuesta en un escenario representativo de producción.

### Conclusión general

La incorporación de un agente de herramientas permitió implementar un mecanismo de selección condicional para determinar cuándo consultar el corpus jurídico. El diagnóstico independiente mostró resultados preliminares favorables, con un 90 % de decisiones correctas y un comportamiento consistente en la generación de respuestas estructuradas.

No obstante, estos resultados deben interpretarse con cautela debido al tamaño limitado de la muestra y a la presencia de falsos negativos en consultas laborales. Se requieren pruebas más amplias para establecer la confiabilidad del agente como controlador de herramientas y evaluar su comportamiento ante situaciones ambiguas.

Por otro lado, la comparación con el sistema RAG avanzado evidenció que la incorporación del agente no produjo mejoras en la calidad promedio de las respuestas ni en la recuperación normativa, y que introdujo una mayor latencia mediana en los casos evaluados.

En consecuencia, el aporte principal del agente no se encuentra en mejorar directamente la precisión del recuperador, sino en ofrecer una arquitectura capaz de ejecutar herramientas de forma selectiva. Su potencial para reducir procesamiento innecesario es prometedor, pero requiere validación adicional. Paralelamente, los resultados sugieren que mejorar la recuperación normativa y analizar la cobertura del corpus son aspectos prioritarios para incrementar la calidad del sistema completo.

Finalmente, como limitaciones queda el probar con mayor exhaustividad, probar diferentes prompts aunque se hicieron unas 8  modificaciones para mejorar el rendimiento del agente, pruebas con mayor datos para determinar una mejor precisión del agente e incluso un posible cambio del modelo, esto si realmente se quiere utilizar la herramienta del agente como un posible ahorro de llamado al sistema completo confiando en su presición o si lo que se prioriza dentro del caso de negocio es la latencia, considerar mejor no utilizarlo por el aumento significativo que se obtuvieron en las pruebas.

la realidad es que el agente no mejora las métricas agregadas y añade latencia. Además, produjo abstenciones indebidas en dos consultas laborales y no evitó la búsqueda en el caso oficial fuera de dominio. Por ello, su utilidad selectiva no quedó demostrada.